In [ ]:
%sql
USE CATALOG purgo_databricks;

-- Databricks SQL script: Generate comprehensive test data for purgo_playground.patient_data_auto_loader table
-- Purpose: Insert diverse test records covering happy path, edge, error, NULL, and special character scenarios
-- Author: Giang Nguyen
-- Date: 2025-09-29
-- Description: This script inserts 25 test records into purgo_playground.patient_data_auto_loader, ensuring all columns except data_loaded_at are STRING, and data_loaded_at is TIMESTAMP. Scenarios include valid, boundary, error, NULL, and special/multibyte character cases.

-- CTE: Generate test data for patient_data_auto_loader
WITH test_patient_data AS (
  SELECT 'P001' AS patient_id, 'John Smith' AS patient_name, '45' AS age, 'Diabetes' AS diagnosis, 'Insulin' AS treatment, CAST('2024-03-21T00:00:00.000+0000' AS TIMESTAMP) AS data_loaded_at -- Happy path
  UNION ALL SELECT 'P002', 'Jane Doe', '38', 'Hypertension', 'Beta Blocker', CAST('2024-03-21T01:00:00.000+0000' AS TIMESTAMP) -- Happy path
  UNION ALL SELECT 'P003', 'Alice Lee', '29', 'Asthma', 'Inhaler', CAST('2024-03-21T02:00:00.000+0000' AS TIMESTAMP) -- Happy path
  UNION ALL SELECT 'P004', 'Bob O''Connor', '65', 'COPD', 'Steroid', CAST('2024-03-21T03:00:00.000+0000' AS TIMESTAMP) -- Special char in name
  UNION ALL SELECT 'P005', '李小龙', '32', '肺炎', '抗生素', CAST('2024-03-21T04:00:00.000+0000' AS TIMESTAMP) -- Multibyte chars
  UNION ALL SELECT 'P006', 'Marie Curie', '0', 'Cancer', 'Radiation', CAST('2024-03-21T05:00:00.000+0000' AS TIMESTAMP) -- Edge: age lower bound
  UNION ALL SELECT 'P007', 'Max Mustermann', '120', 'Arthritis', 'Painkiller', CAST('2024-03-21T06:00:00.000+0000' AS TIMESTAMP) -- Edge: age upper bound
  UNION ALL SELECT 'P008', 'Anna Müller', '', 'Migraine', 'Triptan', CAST('2024-03-21T07:00:00.000+0000' AS TIMESTAMP) -- NULL/empty age
  UNION ALL SELECT 'P009', 'Omar Al-Farooq', '55', '', 'Antiviral', CAST('2024-03-21T08:00:00.000+0000' AS TIMESTAMP) -- NULL/empty diagnosis
  UNION ALL SELECT 'P010', 'Zoë Kravitz', '27', 'Flu', '', CAST('2024-03-21T09:00:00.000+0000' AS TIMESTAMP) -- NULL/empty treatment
  UNION ALL SELECT 'P011', '', '40', 'COVID-19', 'Remdesivir', CAST('2024-03-21T10:00:00.000+0000' AS TIMESTAMP) -- NULL/empty patient_name
  UNION ALL SELECT '', 'Carlos Santana', '50', 'Diabetes', 'Metformin', CAST('2024-03-21T11:00:00.000+0000' AS TIMESTAMP) -- NULL/empty patient_id
  UNION ALL SELECT 'P012', 'Élodie Dupont', 'abc', 'Anxiety', 'Therapy', CAST('2024-03-21T12:00:00.000+0000' AS TIMESTAMP) -- Error: age not numeric but valid string
  UNION ALL SELECT 'P013', 'Müller, Anna', '35', 'Depression', 'SSRI', CAST('2024-03-21T13:00:00.000+0000' AS TIMESTAMP) -- Special char: comma in name
  UNION ALL SELECT 'P014', 'Jean-Pierre', '42', 'Hypertension', 'β-blocker', CAST('2024-03-21T14:00:00.000+0000' AS TIMESTAMP) -- Special char: Greek letter
  UNION ALL SELECT 'P015', 'Núñez', '28', 'Asthma', 'Inhaler', CAST('2024-03-21T15:00:00.000+0000' AS TIMESTAMP) -- Special char: accent
  UNION ALL SELECT 'P016', 'Test Nulls', NULL, NULL, NULL, CAST('2024-03-21T16:00:00.000+0000' AS TIMESTAMP) -- All NULLs except id/name
  UNION ALL SELECT NULL, NULL, NULL, NULL, NULL, CAST('2024-03-21T17:00:00.000+0000' AS TIMESTAMP) -- All NULLs
  UNION ALL SELECT 'P017', 'Duplicate Row', '33', 'Diabetes', 'Insulin', CAST('2024-03-21T18:00:00.000+0000' AS TIMESTAMP) -- Duplicate row 1
  UNION ALL SELECT 'P017', 'Duplicate Row', '33', 'Diabetes', 'Insulin', CAST('2024-03-21T18:00:00.000+0000' AS TIMESTAMP) -- Duplicate row 2
  UNION ALL SELECT 'P018', 'Special!@#$', '44', 'Allergy', 'Antihistamine', CAST('2024-03-21T19:00:00.000+0000' AS TIMESTAMP) -- Special chars in name/treatment
  UNION ALL SELECT 'P019', 'Long Name ' || REPEAT('A', 100), '39', 'Chronic Pain', 'Opioid', CAST('2024-03-21T20:00:00.000+0000' AS TIMESTAMP) -- Edge: long string
  UNION ALL SELECT 'P020', 'Tab\tName', '41', 'Insomnia', 'Melatonin', CAST('2024-03-21T21:00:00.000+0000' AS TIMESTAMP) -- Special char: tab
  UNION ALL SELECT 'P021', 'Newline\nName', '36', 'Anemia', 'Iron', CAST('2024-03-21T22:00:00.000+0000' AS TIMESTAMP) -- Special char: newline
  UNION ALL SELECT 'P022', 'Emoji 😀', '25', 'Stress', 'Yoga', CAST('2024-03-21T23:00:00.000+0000' AS TIMESTAMP) -- Multibyte: emoji
  UNION ALL SELECT 'P023', 'NULL', 'NULL', 'NULL', 'NULL', CAST('2024-03-22T00:00:00.000+0000' AS TIMESTAMP) -- Literal 'NULL' string values
)

-- Insert test data into patient_data_auto_loader
INSERT INTO purgo_playground.patient_data_auto_loader (patient_id, patient_name, age, diagnosis, treatment, data_loaded_at)
SELECT patient_id, patient_name, age, diagnosis, treatment, data_loaded_at
FROM test_patient_data;

-- CTE: Validation query to check inserted test data
-- This CTE selects all test records for validation purposes
WITH validate_patient_data AS (
  SELECT patient_id, patient_name, age, diagnosis, treatment, data_loaded_at
  FROM purgo_playground.patient_data_auto_loader
  WHERE data_loaded_at BETWEEN CAST('2024-03-21T00:00:00.000+0000' AS TIMESTAMP) AND CAST('2024-03-22T00:00:00.000+0000' AS TIMESTAMP)
)
SELECT * FROM validate_patient_data;
